In [2]:
import os 
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
from pydantic import BaseModel, ConfigDict,ValidationError


In [3]:
class CVinfo(BaseModel):
    model_config=ConfigDict(extra="forbid")
    name:str
    skills:list[str]
    years_experience:int
good='{"name":"Priya Sharma","skills":["Python","SQL"],"years_experience":4}'
print(CVinfo.model_validate_json(good))
bad='{"name": "Priya Sharma", "skills": ["Python", "SQL"], "years_experience": "4 years"}'
print(CVinfo.model_validate_json(bad))

name='Priya Sharma' skills=['Python', 'SQL'] years_experience=4


ValidationError: 1 validation error for CVinfo
years_experience
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='4 years', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

In [5]:
from openai import OpenAI
client=OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [6]:
SYSTEM="""Extract facts from the cv text.
name: the person's full name 
skills: every tool programming langauge named , one per item.
years_experience: total years of paid work, as a whole number; 0 if none."""

FORMAT={
    "type":"json_schema",
    "json_schema":{"name":"cv","schema":CVinfo.model_json_schema(),"strict":True}
}

cv="priya sharma has worked as a data analyst for 4 years at infosys she knows python,SQL and tableau,and recently learned Fastapi"
messages=[{"role":"system","content":SYSTEM},{"role":"user","content":cv}]
response=client.chat.completions.create(model="qwen2.5:3b-instruct-q4_0",
messages=messages,response_format=FORMAT)
reply=response.choices[0].message.content
print(reply)
print(CVinfo.model_validate_json(reply))

{
  "name": "priya sharma",
  "skills": ["python", "SQL", "tableau", "fastapi"],
  "years_experience": 4
}
name='priya sharma' skills=['python', 'SQL', 'tableau', 'fastapi'] years_experience=4


In [9]:
def extract(cv_text,max_tries=3):
    messages=[{"role":"system","content":SYSTEM},{"role":"user","content":cv_text}]
    for attempt in range(1,max_tries+1):
        response=client.chat.completions.create(model="qwen2.5:3b-instruct-q4_0",messages=messages,response_format=FORMAT)
        reply=response.choices[0].message.content
        print(response.choices[0].finish_reason)
        try:
            return CVinfo.model_validate_json(reply)
        except ValidationError as e:
            print(f"attempt{attempt} failed {e}")
            messages.append({"role":"assistant","content":reply})
            messages.append({"role":"user","content":f"your reply failed this check {e}.reply again with fixed JSON"})
    raise ValueError("No vaild reply after 3 tries")



In [ ]:
for cv in ["Priya Sharma has worked as a data analyst for 4 years at Infosys. She knows Python, SQL and Tableau, and recently learned FastAPI.",
           "Rahul Verma: backend developer. Java and Spring Boot at TCS from 2019 to 2022, then Go and Kubernetes at Zomato since 2022 (3 years).",
           "Fresh graduate Aisha Khan built two NLP projects with PyTorch and Hugging Face during her MSc. No work experience yet."]:
    print(extract(cv))

stop
name='Priya Sharma' skills=['Python', 'SQL', 'Tableau', 'FastAPI'] years_experience=4
stop
name='Rahul Verma' skills=['Java', 'Spring Boot', 'Go', 'Kubernetes'] years_experience=3
stop
name='Aisha Khan' skills=['PyTorch', 'Hugging Face', 'Natural Language Processing (NLP)'] years_experience=2


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "~/projects/AI engineering /.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "~/projects/AI engineering /.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 584, in shell_channel_thread_main
    _, msg2 = self.session.feed_identities(msg, copy=False)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "~/projects/AI engineering /.venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "~/projects/AI engineering /.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "~/projects/AI engineering /.venv/lib/python3.12/site-packages/ipykernel/kernel